# **1. Crawl danh mục và link sản phẩm**

In [1]:
import requests
import json
import time
import pandas as pd
from typing import List, Dict
from urllib.parse import urlencode
import os

class TikiCategoryCrawler:
    def __init__(self):
        self.headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/142.0.0.0 Safari/537.36 Edg/142.0.0.0",
            "Accept": "application/json, text/plain, */*",
            "Referer": "https://tiki.vn/",
            "x-guest-token": "FGPAnqfVsJMDaElxYiT6z1IyUCWR3Odg"
        }
        self.category_url = "https://tiki.vn/api/v2/categories"
        self.listings_url = "https://tiki.vn/api/personalish/v1/blocks/listings"

        # 5 danh mục lớn cần crawl
        self.main_categories = [
            {"id": 8594, "name": "Ô Tô – Xe Máy – Xe Đạp"},
            {"id": 4384, "name": "Bách Hóa Online"},
            {"id": 1975, "name": "Thể Thao – Dã Ngoại"},
            {"id": 915, "name": "Thời trang nam"},
            {"id": 1846, "name": "Laptop – Máy Vi Tính – Linh kiện"}
        ]

    def get_categories_level1_by_parent(self, parent_id: int):
        """Lấy tất cả danh mục con cấp 1 từ parent_id"""
        params = {"include": "children", "parent_id": parent_id}

        try:
            response = requests.get(self.category_url, params=params, headers=self.headers)
            if response.status_code == 200:
                data = response.json()
                return data.get("data", [])
            else:
                print(f"Lỗi khi lấy danh mục từ parent {parent_id}: {response.status_code}")
                return []
        except Exception as e:
            print(f"Lỗi: {e}")
            return []

    def get_products_from_category(self, category_id: int, category_name: str, url_key: str, root_category_name: str, pages: int = 3, limit_per_page: int = 75):
        """Lấy sản phẩm từ danh mục cấp 1"""
        products = []

        for page in range(1, pages + 1):
            params = {
                "limit": limit_per_page,
                "sort": "top_seller",
                "page": page,
                "urlKey": url_key,
                "category": category_id
            }

            try:
                print(f"Đang lấy trang {page} của danh mục: {category_name}")
                response = requests.get(self.listings_url, params=params, headers=self.headers, timeout=10)

                if response.status_code == 200:
                    data = response.json()
                    page_products = data.get("data", [])

                    if not page_products:
                        print(f"  - Trang {page}: Không có sản phẩm")
                        break

                    for product in page_products:
                        products.append({
                            "product_id": product.get("id"),
                            "product_name": product.get("name"),
                            "product_url": f"https://tiki.vn/{product.get('url_path', '')}",
                            "category_id": category_id,
                            "category_name": category_name,
                            "category_root_name": root_category_name
                        })

                    print(f"  - Trang {page}: Đã lấy {len(page_products)} sản phẩm")

                    # Kiểm tra xem có còn trang tiếp theo không
                    paging = data.get("paging", {})
                    if page >= paging.get("last_page", page):
                        break

                    time.sleep(1)  # Chờ 1 giây giữa các trang

                else:
                    print(f"  - Lỗi khi lấy trang {page}: {response.status_code}")
                    break

            except Exception as e:
                print(f"  - Lỗi: {e}")
                break

        return products

    def crawl_phase1(self, pages_per_category: int = 3, limit_per_page: int = 75):
        """Crawl phase 1 - Lấy thông tin cơ bản sản phẩm từ 5 danh mục lớn"""
        print("=== BẮT ĐẦU PHẦN 1: LẤY THÔNG TIN DANH MỤC VÀ SẢN PHẨM CƠ BẢN ===")

        all_products = []

        for main_cat in self.main_categories:
            print(f"\n=== ĐANG XỬ LÝ DANH MỤC LỚN: {main_cat['name']} (ID: {main_cat['id']}) ===")

            # Lấy tất cả danh mục con cấp 1 từ danh mục lớn
            sub_categories = self.get_categories_level1_by_parent(main_cat["id"])
            print(f"Đã lấy được {len(sub_categories)} danh mục con cấp 1")

            total_subcategories = len(sub_categories)

            for i, category in enumerate(sub_categories, 1):
                print(f"\n[{i}/{total_subcategories}] Đang xử lý danh mục: {category['name']} (ID: {category['id']})")

                # Lấy sản phẩm từ danh mục cấp 1
                products = self.get_products_from_category(
                    category_id=category["id"],
                    category_name=category["name"],
                    url_key=category["url_key"],
                    root_category_name=main_cat["name"],  # Sử dụng tên danh mục lớn
                    pages=pages_per_category,
                    limit_per_page=limit_per_page
                )

                all_products.extend(products)
                print(f"  - Đã lấy {len(products)} sản phẩm từ danh mục {category['name']}")

                # Thời gian chờ giữa các danh mục con
                if i < total_subcategories:
                    time.sleep(1.5)

            # Thời gian chờ giữa các danh mục lớn
            if main_cat != self.main_categories[-1]:
                print(f"\nĐã hoàn thành danh mục {main_cat['name']}, chờ 1.5 giây trước khi chuyển sang danh mục tiếp theo...")
                time.sleep(1.5)

        # Lưu kết quả phase 1
        df_phase1 = pd.DataFrame(all_products)

        # Đảm bảo thứ tự cột
        columns_order = [
            'product_id', 'product_name', 'product_url',
            'category_id', 'category_name', 'category_root_name'
        ]
        df_phase1 = df_phase1[columns_order]

        # Lưu file CSV vào Google Drive
        from google.colab import drive
        drive.mount('/content/drive')

        # Tạo đường dẫn đến thư mục trên Google Drive
        drive_path = '/content/drive/MyDrive/tiki_products_phase1.csv'

        # Lưu file
        df_phase1.to_csv(drive_path, index=False, encoding='utf-8')

        print(f"\n=== KẾT THÚC PHẦN 1 ===")
        print(f"Tổng số sản phẩm đã lấy: {len(all_products)}")
        print(f"Tổng số danh mục lớn đã xử lý: {len(self.main_categories)}")
        print(f"Đã lưu vào file: {drive_path}")

        return all_products

# Chạy Phase 1
if __name__ == "__main__":
    crawler_phase1 = TikiCategoryCrawler()

    # Lấy tất cả danh mục con từ 5 danh mục lớn, mỗi danh mục 3 trang, mỗi trang tối đa 75 sản phẩm
    products_phase1 = crawler_phase1.crawl_phase1(
        pages_per_category=3,
        limit_per_page=75
    )

=== BẮT ĐẦU PHẦN 1: LẤY THÔNG TIN DANH MỤC VÀ SẢN PHẨM CƠ BẢN ===

=== ĐANG XỬ LÝ DANH MỤC LỚN: Ô Tô – Xe Máy – Xe Đạp (ID: 8594) ===
Đã lấy được 6 danh mục con cấp 1

[1/6] Đang xử lý danh mục: Xe máy (ID: 8597)
Đang lấy trang 1 của danh mục: Xe máy
  - Trang 1: Đã lấy 40 sản phẩm
Đang lấy trang 2 của danh mục: Xe máy
  - Trang 2: Đã lấy 40 sản phẩm
Đang lấy trang 3 của danh mục: Xe máy
  - Trang 3: Đã lấy 40 sản phẩm
  - Đã lấy 120 sản phẩm từ danh mục Xe máy

[2/6] Đang xử lý danh mục: Xe điện (ID: 6070)
Đang lấy trang 1 của danh mục: Xe điện
  - Trang 1: Đã lấy 40 sản phẩm
Đang lấy trang 2 của danh mục: Xe điện
  - Trang 2: Đã lấy 40 sản phẩm
Đang lấy trang 3 của danh mục: Xe điện
  - Trang 3: Đã lấy 40 sản phẩm
  - Đã lấy 120 sản phẩm từ danh mục Xe điện

[3/6] Đang xử lý danh mục: Xe đạp (ID: 8431)
Đang lấy trang 1 của danh mục: Xe đạp
  - Trang 1: Đã lấy 40 sản phẩm
Đang lấy trang 2 của danh mục: Xe đạp
  - Trang 2: Đã lấy 40 sản phẩm
Đang lấy trang 3 của danh mục: Xe đạp
  - Tr

# **2. Crawl chi tiết sản phẩm và Id Shop**

In [6]:
import pandas as pd
import requests
import time
import json
from typing import Dict, List, Optional

class TikiProductDetailCrawler:
    def __init__(self):
        self.headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/142.0.0.0 Safari/537.36 Edg/142.0.0.0",
            "Accept": "application/json, text/plain, */*",
            "Referer": "https://tiki.vn/",
            "x-guest-token": "FGPAnqfVsJMDaElxYiT6z1IyUCWR3Odg"
        }
        self.product_api_url = "https://tiki.vn/api/v2/products/{}"

    def extract_spid_from_url(self, product_url: str) -> Optional[str]:
        """Trích xuất spid từ URL sản phẩm"""
        try:
            if 'spid=' in product_url:
                return product_url.split('spid=')[1].split('&')[0]
            return None
        except:
            return None

    def get_product_details(self, product_id: int, product_url: str) -> Dict:
        """Lấy chi tiết sản phẩm từ API"""
        spid = self.extract_spid_from_url(product_url)

        params = {
            "platform": "web",
            "version": "3"
        }
        if spid:
            params["spid"] = spid

        try:
            url = self.product_api_url.format(product_id)
            response = requests.get(url, params=params, headers=self.headers, timeout=10)

            if response.status_code == 200:
                return response.json()
            else:
                print(f"Lỗi khi lấy chi tiết sản phẩm {product_id}: {response.status_code}")
                return None
        except Exception as e:
            print(f"Lỗi khi lấy sản phẩm {product_id}: {e}")
            return None

    def parse_product_data(self, product_data: Dict, product_url: str) -> Dict:
        """Phân tích và trích xuất dữ liệu từ JSON sản phẩm"""
        if not product_data:
            return {}

        # Đếm số lượng video_url
        video_count = 0
        for key, value in product_data.items():
            if 'video_url' in key.lower() and value is not None:
                if isinstance(value, list):
                    video_count += len(value)
                elif isinstance(value, str) and value.strip():
                    video_count += 1
                elif value:
                    video_count += 1

        # Thông tin cơ bản
        result = {
            'price': product_data.get('price'),
            'original_price': product_data.get('original_price'),
            'discount_rate': product_data.get('discount_rate'),
            'quantity_sold': product_data.get('all_time_quantity_sold'),
            'rating_average': product_data.get('rating_average'),
            'review_count': product_data.get('review_count'),
            'is_return_policy': 1 if product_data.get('return_policy') is not None else 0,
            'is_freeship_xtra': self._has_freeship_xtra(product_data),
            'is_authentic': self._is_authentic(product_data),
            'image_count': len(product_data.get('images', [])),
            'video_count': video_count,
            'is_brand': 1 if product_data.get('brand') else 0,
            'brand_name': product_data.get('brand', {}).get('name') if product_data.get('brand') else None,
            'origin': self._get_origin(product_data),
            'spid': self.extract_spid_from_url(product_url),
        }

        # Thông tin shop
        seller_info = product_data.get('current_seller', {})
        result.update({
            'store_id': seller_info.get('id') if seller_info else None,
        })

        return result

    def _has_freeship_xtra(self, product_data: Dict) -> Optional[int]:
        """Kiểm tra có freeship xtra không"""
        tracking_info = product_data.get('tracking_info')
        if not tracking_info or not isinstance(tracking_info, dict):
            return None

        amplitude = tracking_info.get('amplitude')
        if not amplitude or not isinstance(amplitude, dict):
            return None

        is_freeship = amplitude.get('is_freeship_xtra')
        if is_freeship is None:
            return None
        return 1 if is_freeship else 0

    def _is_authentic(self, product_data: Dict) -> Optional[int]:
        """Kiểm tra có chính hãng không"""
        tracking_info = product_data.get('tracking_info')
        if not tracking_info or not isinstance(tracking_info, dict):
            return None

        amplitude = tracking_info.get('amplitude')
        if not amplitude or not isinstance(amplitude, dict):
            return None

        is_authentic = amplitude.get('is_authentic')
        if is_authentic is None:
            return None
        return 1 if is_authentic else 0

    def _get_origin(self, product_data: Dict) -> Optional[str]:
        """Lấy thông tin xuất xứ"""
        specifications = product_data.get('specifications')
        if not specifications or not isinstance(specifications, list):
            return None

        for spec in specifications:
            if not spec or not isinstance(spec, dict):
                continue

            attributes = spec.get('attributes')
            if not attributes or not isinstance(attributes, list):
                continue

            for attr in attributes:
                if not attr or not isinstance(attr, dict):
                    continue

                if attr.get('code') == 'origin':
                    value = attr.get('value')
                    return value if value else None
        return None

    def crawl_phase2(self, input_file: str = '/content/drive/MyDrive/tiki_products_phase1.csv',
                    output_file: str = '/content/drive/MyDrive/tiki_products_phase2.csv',
                    delay: float = 1.0):
        """Crawl Phase 2 - Chi tiết sản phẩm và thông tin shop"""
        print("=== BẮT ĐẦU PHASE 2: CRAWL CHI TIẾT SẢN PHẨM VÀ THÔNG TIN SHOP ===")

        # Mount Google Drive
        try:
            from google.colab import drive
            drive.mount('/content/drive')
            print("Đã mount Google Drive")
        except:
            print("Không thể mount Google Drive, sử dụng đường dẫn local")
            if input_file.startswith('/content/drive/MyDrive/'):
                input_file = 'tiki_products_phase1.csv'
            if output_file.startswith('/content/drive/MyDrive/'):
                output_file = 'tiki_products_phase2.csv'

        # Đọc dữ liệu từ Phase 1
        try:
            df_phase1 = pd.read_csv(input_file)
            print(f"Đã đọc {len(df_phase1)} sản phẩm từ {input_file}")
        except FileNotFoundError:
            print(f"Không tìm thấy file {input_file}. Hãy chạy Phase 1 trước.")
            return

        # Tạo bản sao để thêm dữ liệu mới
        df_phase2 = df_phase1.copy()

        # Thêm các cột mới với giá trị mặc định
        new_columns = [
            'price', 'original_price', 'discount_rate', 'quantity_sold',
            'rating_average', 'review_count', 'is_return_policy', 'is_freeship_xtra',
            'is_authentic', 'image_count', 'video_count', 'is_brand', 'brand_name',
            'origin', 'spid', 'store_id'
        ]

        for col in new_columns:
            df_phase2[col] = None

        total_products = len(df_phase2)
        success_count = 0

        # Crawl toàn bộ sản phẩm
        for index, row in df_phase2.iterrows():
            product_id = row['product_id']
            product_url = row['product_url']

            print(f"[{index + 1}/{total_products}] Đang xử lý sản phẩm: {product_id}")

            product_detail = self.get_product_details(product_id, product_url)

            if product_detail:
                parsed_data = self.parse_product_data(product_detail, product_url)

                # Cập nhật dữ liệu vào DataFrame
                for key, value in parsed_data.items():
                    if key in df_phase2.columns:
                        df_phase2.at[index, key] = value

                success_count += 1
                print(f"  - Đã cập nhật chi tiết sản phẩm")
            else:
                print(f"  - Không lấy được chi tiết sản phẩm {product_id}")

            # Thời gian chờ giữa các request
            if index < total_products - 1:
                time.sleep(delay)

        # Lưu kết quả cuối cùng
        df_phase2.to_csv(output_file, index=False, encoding='utf-8')

        print(f"\n=== KẾT THÚC PHASE 2 ===")
        print(f"Tổng số sản phẩm đã xử lý: {total_products}")
        print(f"Số sản phẩm thành công: {success_count}")
        print(f"Đã lưu file: {output_file}")

# Chạy Phase 2
if __name__ == "__main__":
    crawler_phase2 = TikiProductDetailCrawler()

    # Crawl chi tiết sản phẩm từ file Phase 1
    crawler_phase2.crawl_phase2(
        input_file='/content/drive/MyDrive/tiki_products_phase1.csv',
        output_file='/content/drive/MyDrive/tiki_products_phase2.csv',
        delay=0.5
    )

Streaming output truncated to the last 5000 lines.
  - Đã cập nhật chi tiết sản phẩm
[2837/5333] Đang xử lý sản phẩm: 272047645
  - Đã cập nhật chi tiết sản phẩm
[2838/5333] Đang xử lý sản phẩm: 262839198
  - Đã cập nhật chi tiết sản phẩm
[2839/5333] Đang xử lý sản phẩm: 251282239
  - Đã cập nhật chi tiết sản phẩm
[2840/5333] Đang xử lý sản phẩm: 249175807
  - Đã cập nhật chi tiết sản phẩm
[2841/5333] Đang xử lý sản phẩm: 214661792
  - Đã cập nhật chi tiết sản phẩm
[2842/5333] Đang xử lý sản phẩm: 214657091
  - Đã cập nhật chi tiết sản phẩm
[2843/5333] Đang xử lý sản phẩm: 212971338
  - Đã cập nhật chi tiết sản phẩm
[2844/5333] Đang xử lý sản phẩm: 212971196
  - Đã cập nhật chi tiết sản phẩm
[2845/5333] Đang xử lý sản phẩm: 206103327
  - Đã cập nhật chi tiết sản phẩm
[2846/5333] Đang xử lý sản phẩm: 202818240
  - Đã cập nhật chi tiết sản phẩm
[2847/5333] Đang xử lý sản phẩm: 194804899
  - Đã cập nhật chi tiết sản phẩm
[2848/5333] Đang xử lý sản phẩm: 179407023
  - Đã cập nhật chi tiết 

# 3. **Crawl chi tiết shop**

In [ ]:
import pandas as pd
import requests
import time
import json
from typing import Dict, List, Optional

class TikiShopCrawler:
    def __init__(self):
        self.headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/142.0.0.0 Safari/537.36 Edg/142.0.0.0",
            "Accept": "application/json, text/plain, */*",
            "Referer": "https://tiki.vn/",
            "x-guest-token": "FGPAnqfVsJMDaElxYiT6z1IyUCWR3Odg"
        }
        self.shop_api_url = "https://api.tiki.vn/product-detail/v2/widgets/seller"
        self.performance_api_url = "https://seller-store-api.tiki.vn/ovl-performances/{}"

    def get_shop_details(self, seller_id: int, product_id: int, spid: str) -> Dict:
        """Lấy chi tiết shop từ API"""
        params = {
            "seller_id": seller_id,
            "mpid": product_id,
            "spid": spid,
            "trackity_id": "ac9e12fb-60ff-e1e0-3fd5-8ed0d8d50033",
            "platform": "desktop",
            "version": "3"
        }

        try:
            response = requests.get(self.shop_api_url, params=params, headers=self.headers, timeout=10)

            if response.status_code == 200:
                return response.json()
            else:
                print(f"Lỗi khi lấy chi tiết shop {seller_id}: {response.status_code}")
                return None
        except Exception as e:
            print(f"Lỗi khi lấy shop {seller_id}: {e}")
            return None

    def get_shop_performance(self, store_id: int) -> Dict:
        """Lấy thông tin performance của shop"""
        try:
            url = self.performance_api_url.format(store_id)
            response = requests.get(url, headers=self.headers, timeout=10)

            if response.status_code == 200:
                return response.json()
            else:
                print(f"Lỗi khi lấy performance shop {store_id}: {response.status_code}")
                return None
        except Exception as e:
            print(f"Lỗi khi lấy performance shop {store_id}: {e}")
            return None

    def parse_shop_data(self, shop_data: Dict) -> Dict:
        """Phân tích và trích xuất dữ liệu từ JSON shop"""
        if not shop_data:
            return {}

        result = {}

        # Kiểm tra data và seller
        data = shop_data.get('data')
        if not data or not isinstance(data, dict):
            return {
                'store_name': None,
                'store_review_count': None,
                'total_follower': None,
                'is_official': None
            }

        seller = data.get('seller')
        if not seller or not isinstance(seller, dict):
            return {
                'store_name': None,
                'store_review_count': None,
                'total_follower': None,
                'is_official': None
            }

        # Xử lý is_official: có = 1, không = 0, không có = None
        is_official = seller.get('is_official')
        if is_official is None:
            is_official_value = None
        else:
            is_official_value = 1 if is_official else 0

        result = {
            'store_name': seller.get('name'),
            'store_review_count': seller.get('review_count'),
            'total_follower': seller.get('total_follower'),
            'is_official': is_official_value
        }

        return result

    def parse_performance_data(self, performance_data: Dict) -> Dict:
        """Phân tích và trích xuất dữ liệu từ JSON performance"""
        if not performance_data:
            return {}

        result = {
            'cancel_by_seller_rate': performance_data.get('cancel_by_seller_rate_l4w'),
            'cancel_by_seller_rate_status': performance_data.get('cancel_by_seller_rate_l4w_status'),
            'return_rate': performance_data.get('return_rate_l4w'),
            'return_rate_status': performance_data.get('return_rate_l4w_status')
        }

        return result

    def crawl_phase3(self, input_file: str = '/content/drive/MyDrive/tiki_products_phase2.csv',
                    output_file: str = '/content/drive/MyDrive/tiki_products.csv',
                    delay: float = 1.0):
        """Crawl Phase 3 - Thông tin chi tiết shop và performance"""
        print("=== BẮT ĐẦU PHASE 3: CRAWL THÔNG TIN SHOP VÀ PERFORMANCE ===")

        # Mount Google Drive
        try:
            from google.colab import drive
            drive.mount('/content/drive')
            print("Đã mount Google Drive")
        except:
            print("Không thể mount Google Drive, sử dụng đường dẫn local")
            if input_file.startswith('/content/drive/MyDrive/'):
                input_file = 'tiki_products_phase2.csv'
            if output_file.startswith('/content/drive/MyDrive/'):
                output_file = 'tiki_products.csv'

        # Đọc dữ liệu từ Phase 2
        try:
            df_phase2 = pd.read_csv(input_file)
            print(f"Đã đọc {len(df_phase2)} sản phẩm từ {input_file}")
        except FileNotFoundError:
            print(f"Không tìm thấy file {input_file}. Hãy chạy Phase 2 trước.")
            return

        # Tạo bản sao để thêm dữ liệu mới
        df_phase3 = df_phase2.copy()

        # Thêm các cột mới với giá trị mặc định
        new_columns = [
            'store_name', 'store_review_count', 'total_follower', 'is_official',
            'cancel_by_seller_rate', 'cancel_by_seller_rate_status',
            'return_rate', 'return_rate_status'
        ]

        for col in new_columns:
            df_phase3[col] = None

        total_products = len(df_phase3)
        success_count = 0

        # Cache để tránh gọi API trùng lặp cho cùng store_id
        performance_cache = {}

        # Crawl toàn bộ sản phẩm
        for index, row in df_phase3.iterrows():
            store_id = row['store_id']
            product_id = row['product_id']
            spid = row['spid']

            print(f"[{index + 1}/{total_products}] Đang xử lý shop: {store_id}")

            # Chỉ crawl nếu có đủ thông tin
            if pd.notna(store_id) and pd.notna(product_id) and pd.notna(spid):
                # Lấy thông tin shop cơ bản
                shop_detail = self.get_shop_details(int(store_id), int(product_id), str(spid))

                if shop_detail:
                    parsed_shop_data = self.parse_shop_data(shop_detail)

                    # Cập nhật dữ liệu shop vào DataFrame
                    for key, value in parsed_shop_data.items():
                        if key in df_phase3.columns:
                            df_phase3.at[index, key] = value

                    # Lấy thông tin performance (chỉ lấy 1 lần cho mỗi store_id)
                    if store_id not in performance_cache:
                        performance_detail = self.get_shop_performance(int(store_id))
                        if performance_detail:
                            parsed_performance_data = self.parse_performance_data(performance_detail)
                            performance_cache[store_id] = parsed_performance_data
                        else:
                            performance_cache[store_id] = {}

                    # Cập nhật dữ liệu performance từ cache
                    performance_data = performance_cache[store_id]
                    for key, value in performance_data.items():
                        if key in df_phase3.columns:
                            df_phase3.at[index, key] = value

                    success_count += 1
                    print(f"  - Đã cập nhật thông tin shop và performance: {parsed_shop_data.get('store_name', 'N/A')}")
                else:
                    print(f"  - Không lấy được thông tin shop {store_id}")
            else:
                print(f"  - Thiếu thông tin để crawl shop (store_id: {store_id}, product_id: {product_id}, spid: {spid})")

            # Thời gian chờ giữa các request
            if index < total_products - 1:
                time.sleep(delay)

        # Xóa cột spid
        if 'spid' in df_phase3.columns:
            df_phase3 = df_phase3.drop(columns=['spid'])
            print("Đã xóa cột spid")

        # Lưu kết quả cuối cùng
        df_phase3.to_csv(output_file, index=False, encoding='utf-8')

        print(f"\n=== KẾT THÚC PHASE 3 ===")
        print(f"Tổng số sản phẩm đã xử lý: {total_products}")
        print(f"Số shop thành công: {success_count}")
        print(f"Đã lưu file: {output_file}")

# Chạy Phase 3
if __name__ == "__main__":
    crawler_phase3 = TikiShopCrawler()

    # Crawl thông tin shop từ file Phase 2
    crawler_phase3.crawl_phase3(
        input_file='/content/drive/MyDrive/tiki_products_phase2.csv',
        output_file='/content/drive/MyDrive/tiki_products.csv',
        delay=2.0
    )

Streaming output truncated to the last 5000 lines.
[2837/5333] Đang xử lý shop: 108828.0
  - Đã cập nhật thông tin shop và performance: Tiện ích 1708
[2838/5333] Đang xử lý shop: 172234.0
  - Đã cập nhật thông tin shop và performance: TOKDODO Official Store
[2839/5333] Đang xử lý shop: 161884.0
  - Đã cập nhật thông tin shop và performance: Quân Camp
[2840/5333] Đang xử lý shop: 325737.0
  - Đã cập nhật thông tin shop và performance: ZIPPO Flagship Store
[2841/5333] Đang xử lý shop: 253455.0
  - Đã cập nhật thông tin shop và performance: CỬA HÀNG 108
[2842/5333] Đang xử lý shop: 253455.0
  - Đã cập nhật thông tin shop và performance: CỬA HÀNG 108
[2843/5333] Đang xử lý shop: 253455.0
  - Đã cập nhật thông tin shop và performance: CỬA HÀNG 108
[2844/5333] Đang xử lý shop: 253455.0
  - Đã cập nhật thông tin shop và performance: CỬA HÀNG 108
[2845/5333] Đang xử lý shop: 158877.0
  - Đã cập nhật thông tin shop và performance: Liên Minh 1974
[2846/5333] Đang xử lý shop: 6862.0
  - Đã cập nh